In [19]:
# TV Show Data Preprocessing
# Input:  data/tv_shows_scraped.pkl
# Output: data/tv_shows.pkl, data/tv_similarity.pkl

import pandas as pd
import pickle
import string
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
except:
    import nltk
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))

stemmer = SnowballStemmer('english')
print('Imports ready')


Imports ready


## 1. Load Data


In [20]:
# Load scraped TV show data
df = pd.read_pickle('data/tv_shows_scraped.pkl')
print(f'Loaded {len(df)} TV shows')
print(f'Columns: {list(df.columns)}')
df.head()


Loaded 7335 TV shows
Columns: ['id', 'imdb_id', 'title', 'year', 'vote_average', 'vote_count', 'popularity', 'number_of_seasons', 'number_of_episodes', 'status', 'overview', 'tagline', 'poster_url', 'genres', 'actors', 'creators', 'keywords', 'watch_providers']


,id,imdb_id,title,year,vote_average,vote_count,popularity,number_of_seasons,number_of_episodes,status,overview,tagline,poster_url,genres,actors,creators,keywords,watch_providers
0,208493,tt27417996,I Got a Cheat Skill in Another World and Becam...,2023.0,8.200,273,21.4045,2,13,Ended,A door to another world stretches out before a...,,https://image.tmdb.org/t/p/w500//dXrUejdxJFgk6...,"[Animation, Action & Adventure, Sci-Fi & Fantasy]","[Yoshitsugu Matsuoka, Akari Kito, Kaori Maeda,...",[],"[elves, magic, high school, bullying, harem, r...","{'US': {'flatrate': [{'id': 283, 'name': 'Crun..."
1,17035,tt0995832,Generation Kill,2008.0,7.911,480,4.8807,1,7,Ended,The first 40 days of the war in Iraq as seen t...,,https://image.tmdb.org/t/p/w500//wiihoYOODwh82...,"[War & Politics, Drama, Action & Adventure]","[Alexander Skarsgård, James Ransone, Lee Terge...",[David Simon],"[based on novel or book, miniseries, iraq war,...","{'US': {'flatrate': [{'id': 1899, 'name': 'HBO..."
2,93221,tt9140632,The Great North,2021.0,6.953,75,17.5604,5,97,Canceled,Follow the Alaskan adventures of the Tobin fam...,It's great up here.,https://image.tmdb.org/t/p/w500//7Ww56dPIuXn4N...,"[Animation, Comedy]","[Nick Offerman, Jenny Slate, Will Forte, Aparn...","[Wendy Molyneux, Lizzie Molyneux-Logelin, Mint...","[family, adult animation, amused]","{'US': {'flatrate': [{'id': 15, 'name': 'Hulu'..."
3,2538,tt0481449,Hyperdrive,2006.0,6.500,30,1.7391,2,12,Ended,"Set in 2151 and 2152, it follows the crew of H...",,https://image.tmdb.org/t/p/w500//zUpHbyVK0O32K...,"[Comedy, Sci-Fi & Fantasy]","[Nick Frost, Miranda Hart, Kevin Eldon, Dan An...","[Andy Riley, Kevin Cecil]","[future, space, sitcom]","{'US': {'flatrate': [{'id': 9, 'name': 'Amazon..."
4,115857,tt11937732,Daughter from Another Mother,2021.0,7.600,1250,7.8773,3,27,Ended,After realizing their babies were exchanged at...,,https://image.tmdb.org/t/p/w500//lq0n5QzBA78nd...,"[Comedy, Drama]","[Ludwika Paleta, Paulina Goto, Martín Altomaro...","[Carolina Rivera, Fernando Sariñana]","[mother, female friendship, daughter, motherho...","{'US': {'flatrate': [{'id': 8, 'name': 'Netfli..."


## 2. Process Features


In [21]:
# Ensure list columns are proper lists
for col in ['genres', 'actors', 'creators', 'keywords']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])
df['overview'] = df['overview'].fillna('')
df['tagline'] = df['tagline'].fillna('')
print('Data cleaned')


Data cleaned


In [22]:
# Process text and names
def process_text(text):
    if not text or not isinstance(text, str): return []
    text = text.lower()
    translator = str.maketrans('', '', string.punctuation)
    words = text.translate(translator).split()
    return [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]

def process_names(names):
    if not isinstance(names, list): return []
    return [name.lower().replace(' ', '') for name in names if name]

df['overview_processed'] = df['overview'].apply(process_text)
df['tagline_processed'] = df['tagline'].apply(process_text)
df['actors_processed'] = df['actors'].apply(process_names)
df['creators_processed'] = df['creators'].apply(process_names)
df['genres_processed'] = df['genres'].apply(lambda x: [g.lower() for g in x] if isinstance(x, list) else [])
df['keywords_processed'] = df['keywords'].apply(lambda x: [stemmer.stem(k.lower().replace(' ', '')) for k in x] if isinstance(x, list) else [])
print('Features processed')


Features processed


## 3. Create Soup


In [ ]:
# Create soup (combined features)
def create_soup(row):
    parts = []
    # Rating: Granular half-point buckets, high weight (quality matters)
    if pd.notna(row.get('vote_average')):
        rating = row['vote_average']
        bucket = int(rating * 2) / 2  # Rounds to nearest 0.5 (e.g., 8.3 -> 8.5, 8.7 -> 8.5)
        parts.extend([f"rating{bucket}"] * 3)  # High weight for quality
    
    # Genres: Important for basic categorization, medium weight
    parts.extend(row.get('genres_processed', []) * 2)
    
    # Overview: MOST IMPORTANT - this is the plot/story, highest weight
    parts.extend(row.get('overview_processed', []) * 3)  # Increased from 1
    
    # Keywords: Important for themes/concepts, high weight
    parts.extend(row.get('keywords_processed', []) * 2)  # Increased from 1
    
    # Tagline: Additional plot context, medium weight
    parts.extend(row.get('tagline_processed', []) * 1)  # Keep at 1
    
    # Actors/Creators: Lower weight (reduced from 2x to 1x)
    parts.extend(row.get('actors_processed', []) * 1)  # Reduced from 2
    parts.extend(row.get('creators_processed', []) * 1)  # Reduced from 2
    
    return ' '.join(parts)

df['soup'] = df.apply(create_soup, axis=1)
print('Soup created')


Soup created


## 4. Compute TF-IDF & Similarity


In [24]:
# TF-IDF with optimized settings
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=3, max_df=0.8, max_features=20000, stop_words='english', dtype='float32')
print('Fitting TF-IDF...')
tfidf_matrix = tf.fit_transform(df['soup'].values.astype(str))
print(f'TF-IDF shape: {tfidf_matrix.shape}')


Fitting TF-IDF...


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


TF-IDF shape: (7335, 19796)


In [25]:
# Cosine similarity
print('Computing similarity...')
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Similarity shape: {cosine_sim.shape}')


Computing similarity...
Similarity shape: (7335, 7335)


## 5. Test Recommendations


In [26]:
# Test recommendations
df = df.reset_index(drop=True)
indices = pd.Series(df.index, index=df['title'])

def get_recommendations(title, n=10):
    if title not in indices: return None
    idx = indices[title]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    return df[['title', 'year', 'vote_average']].iloc[[i[0] for i in sim_scores]]

get_recommendations('Breaking Bad', 10)


,title,year,vote_average
6268,Better Call Saul,2015.0,8.702
5046,Pluribus,2025.0,7.968
4165,El Chapo,2017.0,7.524
2859,The Girls at the Back,2022.0,7.550
2522,Narcos,2015.0,8.060
2372,Het gouden uur,2022.0,7.300
2166,Okupas,2000.0,8.597
4140,Your Honor,2020.0,8.072
5150,In Plain Sight,2008.0,7.000
3438,Westworld,2016.0,8.027


## 6. Save Output


In [27]:
# Save output
with open('data/tv_shows.pkl', 'wb') as f:
    pickle.dump(df, f)
with open('data/tv_similarity.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)
print(f'Done! Saved {len(df)} TV shows.')


Done! Saved 7335 TV shows.
